In [1]:
import phonlp
from graph.src.triplet_extraction import init_vncorenlp

vncorenlp_client = init_vncorenlp(r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-1.2")
phoNLP_model = phonlp.load(save_dir=r"E:\Github\uit_chatbot\graph\phonlp")

Loading model from: E:\Github\uit_chatbot\graph\phonlp/phonlp.pt


In [2]:
from graph.src.db import init_sqlite

process_conn, process_cursor = init_sqlite(r"E:\Github\uit_chatbot\graph\jupyter\uit_law.db")

In [11]:
def parse_dataframe_to_tokens(df):
    """
    Convert DataFrame to a list of token dicts.
    """
    tokens = []
    for _, row in df.iterrows():
        token = {
            'id': int(row['id']),
            'word': str(row['word']),
            'pos': str(row['pos']),
            'head': int(row['head']),
            'deprel': str(row['deprel'])
        }
        tokens.append(token)
    return tokens


def split_sentence_np_vp(tokens):
    if not tokens:
        return [], []

    root_index = -1
    root_id = None

    # Step 1: find the main verb (root or first valid verb)
    for i, token in enumerate(tokens):
        if token['pos'] == 'V':
            if token['deprel'] == 'root' and token['head'] == 0:
                # Avoid picking verb at start (index 0)
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']
                break
            elif root_index == -1 and token['deprel'] != 'nmod':
                # Avoid first word if it's a verb
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']

    # Step 2: fallback – pick next verb if root not found
    if root_index == -1:
        for i, token in enumerate(tokens):
            if token['pos'] == 'V' and i > 0:  # skip first position
                root_index = i
                root_id = token['id']
                break

    # Step 3: final split
    if root_index != -1 and root_id is not None:
        np_tokens = tokens[:root_index]
        vp_tokens = tokens[root_index:]
        return np_tokens, vp_tokens

    return [], []

def collect_dependents(tokens, head_id):
    """Return set of token ids: head_id + all recursive dependents"""
    subtree = {head_id}
    result = []
    added = True
    while added:
        added = False
        for token in tokens:
            if token['head'] in subtree and token['id'] not in subtree:
                subtree.add(token['id'])
                result.append(token)
                added = True
    return result


def collect_direct_dependents(tokens, head_id):
    """Return list of token dicts that directly depend on head_id"""
    return [t for t in tokens if t['head'] == head_id]


def rebuild_phrase(tokens):
    # Sort tokens by their original position in the sentence and join them together
    tokens_sorted = sorted(tokens, key=lambda x: x['id'])
    phrase = " ".join(t['word'] for t in tokens_sorted)
    return phrase

def extract_main_subjects(np_tokens):
    if not np_tokens:
        return []

    sub_tokens = [t for t in np_tokens if t['deprel'] == 'sub']
    if not sub_tokens:
        sub_tokens = [t for t in np_tokens if t['deprel'] == 'root']
    if not sub_tokens:
        return []

    main_subjects = [sub_tokens[0]]
    main_subjects.extend(collect_direct_dependents(np_tokens, sub_tokens[0]['id']))
    if len(main_subjects) == len(np_tokens):
        return [rebuild_phrase(np_tokens)]

    # Find Coordination Word (Cc, CH)
    coord_tokens = [t for t in np_tokens if t['pos'] in ['Cc', 'CH']]

    if len(coord_tokens) > 0:
        phrases = []

        for coord in coord_tokens:
            left_tokens = []
            temp = -1
            for i, token in enumerate(np_tokens):
                if token['id'] < coord['id'] and token['pos'].startswith('N') and token['deprel'] != 'sub':
                    temp = i

            if temp != -1:
                left_tokens.append(np_tokens[temp])
                left_tokens.extend(collect_direct_dependents(np_tokens, np_tokens[temp]['id']))

            if left_tokens:
                phrases.append(rebuild_phrase(left_tokens))

            right_tokens = []
            coord_dependents = collect_direct_dependents(np_tokens, coord['id'])
            for token in coord_dependents:
                right_tokens.extend(collect_direct_dependents(np_tokens, token['id']))
                right_tokens.append(token)

            if right_tokens:
                phrases.append(rebuild_phrase(right_tokens))

        # Remove duplicates while preserving order
        phrases = list(dict.fromkeys(phrases))
        main_subject_phrase = rebuild_phrase(main_subjects)

        # Properly combine main subject with each phrase
        combined_phrases = []
        for phrase in phrases:
            combined_phrases.append(main_subject_phrase + " " + phrase)
        return combined_phrases
    else:
        return [rebuild_phrase(np_tokens)]


def extract_main_verb(vp_tokens):
    if not vp_tokens:
        return None, []

    # Find the root verb first
    root_verb = None
    for t in vp_tokens:
        if t['deprel'] == 'root' and t['head'] == 0 and t['pos'] == 'V':
            root_verb = t
            break

    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V' and t['deprel'] not in ['nmod', 'aux']:
                root_verb = t
                break

    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V':
                root_verb = t
                break

    # Fallback: first token
    if not root_verb:
        return vp_tokens[0]['word'], [vp_tokens[0]]

    # Get root's direct dependents
    root_dependents = collect_direct_dependents(vp_tokens, root_verb['id'])
    root_dependent_ids = {t['id'] for t in root_dependents}

    # Find coordination words (Cc, CH) that are root or root's direct dependents
    coord_tokens = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH'] and
                    (t['id'] == root_verb['id'] or t['id'] in root_dependent_ids)]

    # If coordination exists, find coordinated verbs
    if coord_tokens:
        verbs = [root_verb]
        all_tokens = [root_verb]

        # Find verbs that are dependents of these coordination tokens
        for coord in coord_tokens:
            coord_dependents = [t for t in vp_tokens if t['head'] == coord['id'] and t['pos'] == 'V']
            verbs.extend(coord_dependents)

        # Remove duplicates while preserving order
        seen_ids = set()
        unique_verbs = []
        for v in verbs:
            if v['id'] not in seen_ids:
                seen_ids.add(v['id'])
                unique_verbs.append(v)
        verbs = unique_verbs

        verb_phrases = []
        for verb in verbs:
            phrase_tokens = [verb]
            dependents = collect_direct_dependents(vp_tokens, verb['id'])
            phrase_tokens.extend(dependents)
            all_tokens.extend(phrase_tokens)
            verb_phrases.append(rebuild_phrase(phrase_tokens))

        # Add coordination tokens
        all_tokens.extend(coord_tokens)

        # Combine with coordination words in order
        result_parts = []
        verb_idx = 0
        for token in sorted(vp_tokens, key=lambda x: x['id']):
            if token in verbs and verb_idx < len(verb_phrases):
                result_parts.append(verb_phrases[verb_idx])
                verb_idx += 1
            elif token['pos'] in ['Cc', 'CH'] and token in coord_tokens:
                result_parts.append(token['word'])

        return ' '.join(result_parts), all_tokens

    # Single verb: return it with its dependents
    verb_tokens = [root_verb]
    verb_tokens.extend(collect_direct_dependents(vp_tokens, root_verb['id']))
    sorted_verb_tokens = sorted(verb_tokens, key=lambda x: x['id'])

    # Filter out tokens after the first noun
    filtered_tokens = []
    for token in sorted_verb_tokens:
        if token['pos'].startswith('N'):
            break
        filtered_tokens.append(token)

    # If we filtered out everything, at least return the root verb
    if not filtered_tokens:
        filtered_tokens = [root_verb]

    return rebuild_phrase(filtered_tokens), filtered_tokens

def extract_objects(np_tokens, verb_token):
    for v in verb_token:
        for t in np_tokens:
            if t['id'] == v['id']:
                np_tokens.remove(t)
                break

    if not np_tokens:
        return []

    # Find object tokens (dob = direct object, iob = indirect object marker, pob = prepositional object)
    obj_tokens = [t for t in np_tokens if t['deprel'] in ['dob', 'iob', 'pob']]
    if not obj_tokens:
        return []

    main_objects = [obj_tokens[0]]
    main_objects.extend(collect_direct_dependents(np_tokens, obj_tokens[0]['id']))
    if len(main_objects) == len(np_tokens):
        return [rebuild_phrase(np_tokens)]

    # Find Coordination Word (Cc, CH)
    coord_tokens = [t for t in np_tokens if t['pos'] in ['Cc', 'CH']]

    if len(coord_tokens) > 0:
        phrases = []

        for coord in coord_tokens:
            left_tokens = []
            temp = -1
            for i, token in enumerate(np_tokens):
                if token['id'] < coord['id'] and token['pos'].startswith('N') and token['deprel'] not in ['dob', 'iob', 'pob']:
                    temp = i

            if temp != -1:
                left_tokens.append(np_tokens[temp])
                left_tokens.extend(collect_direct_dependents(np_tokens, np_tokens[temp]['id']))

            right_tokens = []
            coord_dependents = collect_direct_dependents(np_tokens, coord['id'])
            for token in coord_dependents:
                if token['pos'] == 'N':
                    right_tokens.extend(collect_direct_dependents(np_tokens, token['id']))
                    right_tokens.append(token)
                elif token['pos'] == 'V':
                    left_tokens.append(token)
                    left_tokens.extend(collect_dependents(np_tokens, token['id']))

            if left_tokens:
                phrases.append(rebuild_phrase(left_tokens))
            if right_tokens:
                phrases.append(rebuild_phrase(right_tokens))

        # Remove duplicates while preserving order
        phrases = list(dict.fromkeys(phrases))
        main_object_phrase = rebuild_phrase(main_objects)

        # Properly combine main object with each phrase
        combined_phrases = []
        for phrase in phrases:
            combined_phrases.append(main_object_phrase + " " + phrase)
        return combined_phrases
    else:
        return [rebuild_phrase(np_tokens)]


def process_sentence(df):
    tokens = parse_dataframe_to_tokens(df)
    np_tokens, vp_tokens = split_sentence_np_vp(tokens)

    # Debug prints
    print("-----------------NP-----------------")
    print(np_tokens)
    print("-----------------VP-----------------")
    print(vp_tokens)

    # Step 3: extract subjects, verbs, objects
    subjects = extract_main_subjects(np_tokens)    # list of phrases
    verbs, verbs_token = extract_main_verb(vp_tokens)          # list of verbs
    objects = extract_objects(vp_tokens, verbs_token)          # list of object phrases

    print("-----------------subjects----------------")
    print(subjects)
    print("-----------------verbs----------------")
    print(verbs)
    print("-----------------objects----------------")
    print(objects)

    # Ensure all are lists
    if not isinstance(subjects, list):
        subjects = [subjects]
    if not isinstance(verbs, list):
        verbs = [verbs] if verbs else []
    if not isinstance(objects, list):
        objects = [objects] if objects else []

    # Step 4: combine them into triplets
    triplets = []
    for subj in subjects:
        for verb in verbs:
            for obj in objects:
                triplets.append((subj, verb, obj))

    return triplets

from graph.src.triplet_extraction import load_stopwords, is_valid_term, parsing_result
from graph.src.triplet_extraction.pos_taging.utils import clean_text

stopwords = load_stopwords(r"E:\Github\uit_chatbot\graph\stopwords.csv")

text = "Chương trình đào tạo của mỗi ngành đào tạo do trường xây dựng phù hợp với các quy định hiện hành của Bộ GD&ĐT và ĐHQG-HCM, được bổ sung nội dung xây dựng kế hoạch và thực hiện các điều kiện đảm bảo chất lượng giáo dục"

sentence = clean_text(text)
segmented_text = vncorenlp_client.word_segment(sentence)

# Stopword filtering
parts = segmented_text[0].split(" ")
filtered_parts = [part for part in parts if is_valid_term(part, stopwords)]
filtered_text = " ".join(filtered_parts)
print(segmented_text[0])

# Annotate the filtered text
annotation = phoNLP_model.annotate(text=segmented_text[0])
# annotation = phoNLP_model.annotate(text=filtered_text)
df = parsing_result(annotation)
print(df.to_string(index=False))
result = process_sentence(df)
print("Extracted Triplet:")
for r in result:
    print(r)

chương_trình đào_tạo của mỗi ngành đào_tạo do trường xây_dựng phù_hợp với các quy_định hiện_hành của bộ gd đt và đhqg-hcm , được bổ_sung nội_dung xây_dựng kế_hoạch và thực_hiện các điều_kiện đảm_bảo chất_lượng giáo_dục


100%|██████████| 1/1 [00:00<00:00,  9.48it/s]

 id         word pos head deprel
  1 chương_trình   N   10    sub
  2      đào_tạo   V    1   nmod
  3          của   E    1   nmod
  4          mỗi   L    5    det
  5        ngành   N    3    pob
  6      đào_tạo   V    5   nmod
  7           do   E   10    prp
  8       trường   N    9    sub
  9     xây_dựng   V    7    dep
 10      phù_hợp   V    0   root
 11          với   E   10   vmod
 12          các   L   13    det
 13     quy_định   V   11    pob
 14    hiện_hành   V   13   nmod
 15          của   E   13   nmod
 16           bộ   N   15    pob
 17           gd  Ny   16   nmod
 18           đt  Ny   17   nmod
 19           và  Cc   17  coord
 20     đhqg-hcm  Ny   17   nmod
 21            ,  CH   10  punct
 22         được   V   10   vmod
 23      bổ_sung   V   10   vmod
 24     nội_dung   N   23    dob
 25     xây_dựng   V   24   nmod
 26     kế_hoạch   N   25    dob
 27           và  Cc   25  coord
 28    thực_hiện   V   27   conj
 29          các   L   30    det
 30    điề

In [68]:
from graph.src.triplet_extraction import clean_text
from graph.src.db import extract_random_rows

row = extract_random_rows(process_cursor, "laws_process", limit=1)[0]
_id = row["id"]
so_hieu = row["so_hieu"]
sentence = row["content"]
print(_id)
print(so_hieu)
print(sentence)

3b20300ff16c9e3b29dfc8d57408835f182fb4012ec222b57a6f433a686cf5a0
790/QĐ-ĐHCNTT
Chương trình đào tạo của mỗi ngành đào tạo do trường xây dựng phù hợp với các quy định hiện hành của Bộ GD&ĐT và ĐHQG-HCM, được bổ sung nội dung xây dựng kế hoạch và thực hiện các điều kiện đảm bảo chất lượng giáo dục


In [71]:
from graph.src.triplet_extraction import load_stopwords, is_valid_term, parsing_result
from graph.src.triplet_extraction.pos_taging.utils import clean_text

stopwords = load_stopwords(r"E:\Github\uit_chatbot\graph\stopwords.csv")

# text = "Kế hoạch học tập của mỗi học kỳ và năm học bao gồm thời gian biểu và được công bố trên trang thông tin điện tử của trường"
text = sentence

sentence = clean_text(text)
segmented_text = vncorenlp_client.word_segment(sentence)

# Stopword filtering
print(segmented_text[0])
parts = segmented_text[0].split(" ")
filtered_parts = [part for part in parts if is_valid_term(part, stopwords)]
filtered_text = " ".join(filtered_parts)

# Annotate the filtered text
annotation = phoNLP_model.annotate(text=segmented_text[0])
# annotation = phoNLP_model.annotate(text=filtered_text)
df = parsing_result(annotation)
print(df.to_string(index=False))
result = process_sentence(df)
print("Extracted Triplet:")
for r in result:
    print(r)

chương_trình đào_tạo của mỗi ngành đào_tạo do trường xây_dựng phù_hợp với các quy_định hiện_hành của bộ gd & đt và đhqg-hcm , được bổ_sung nội_dung xây_dựng kế_hoạch và thực_hiện các điều_kiện đảm_bảo chất_lượng giáo_dục


100%|██████████| 1/1 [00:00<00:00,  7.73it/s]

 id         word pos head deprel
  1 chương_trình   N   10    sub
  2      đào_tạo   V    1   nmod
  3          của   E    1   nmod
  4          mỗi   L    5    det
  5        ngành   N    3    pob
  6      đào_tạo   V    5   nmod
  7           do   E   10    prp
  8       trường   N    9    sub
  9     xây_dựng   V    7    dep
 10      phù_hợp   V    0   root
 11          với   E   10   vmod
 12          các   L   13    det
 13     quy_định   V   11    pob
 14    hiện_hành   V   13   nmod
 15          của   E   13   nmod
 16           bộ   N   15    pob
 17           gd  Ny   16   nmod
 18            &  CH   17  punct
 19           đt  Ny   17   nmod
 20           và  Cc   17  coord
 21     đhqg-hcm  Ny   17   nmod
 22            ,  CH   10  punct
 23         được   V   10   vmod
 24      bổ_sung   V   10   vmod
 25     nội_dung   N   24    dob
 26     xây_dựng   V   25   nmod
 27     kế_hoạch   N   26    dob
 28           và  Cc   26  coord
 29    thực_hiện   V   28   conj
 30       